<a href="https://colab.research.google.com/github/moizr1732/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

%pip -q install duckdb

import duckdb
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

DEV_MONTH = "2026-03"
PRIOR_MONTH = "2026-02"

FACT_DEV   = f"read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={DEV_MONTH}/*.parquet')"
FACT_PRIOR = f"read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={PRIOR_MONTH}/*.parquet')"
DIM_CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"

con.sql(f"DESCRIBE SELECT * FROM {FACT_DEV} LIMIT 1").show()
con.sql(f"DESCRIBE SELECT * FROM {DIM_CONTENT} LIMIT 1").show()



┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [31]:
con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT content_hash_id) AS n_content_items,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {FACT_DEV}
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬─────────────────┬────────────┬────────────┐
│ n_rows  │ n_content_items │  min_date  │  max_date  │
│  int64  │      int64      │    date    │    date    │
├─────────┼─────────────────┼────────────┼────────────┤
│ 9841378 │          331437 │ 2026-03-01 │ 2026-03-31 │
└─────────┴─────────────────┴────────────┴────────────┘



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [32]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


label_df = con.sql(f"""
    SELECT content_hash_id,
           SUM(gsc_clicks)       AS march_clicks,
           SUM(gsc_impressions)  AS march_impressions,
           SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS march_ctr
    FROM {FACT_DEV}
    GROUP BY content_hash_id
""").df()

label_df.shape, label_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

((331437, 4),
             content_hash_id  march_clicks  march_impressions  march_ctr
 0  content_b7e512995f79d5a6           2.0             1140.0   0.001754
 1  content_05597932fe4da067           0.0               57.0   0.000000
 2  content_905aa32a0230694e           0.0              149.0   0.000000
 3  content_05434271b257bb68           6.0             1421.0   0.004222
 4  content_d056587ff7faca0c          16.0             2770.0   0.005776)

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.%pip -q install duckdb
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT_DEV}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Rows violating the stated grain (should be 0):", len(grain_check))
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating the stated grain (should be 0): 0


,report_date,client_hash_id,content_hash_id,c


In [34]:
con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT content_hash_id) AS n_content_items,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {FACT_DEV}
""").show()

┌─────────┬─────────────────┬────────────┬────────────┐
│ n_rows  │ n_content_items │  min_date  │  max_date  │
│  int64  │      int64      │    date    │    date    │
├─────────┼─────────────────┼────────────┼────────────┤
│ 9841378 │          331437 │ 2026-03-01 │ 2026-03-31 │
└─────────┴─────────────────┴────────────┴────────────┘



In [35]:
avail = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)  AS ga4_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS ga4_unavailable_or_null_rows
    FROM {FACT_DEV}
""").df()

avail["pct_available"] = (avail["ga4_available_rows"] / avail["total_rows"] * 100).round(1)
avail

,total_rows,ga4_available_rows,ga4_unavailable_or_null_rows,pct_available
0,9841378,413966,9427412,4.2


In [36]:
feb_agg = con.sql(f"""
    SELECT content_hash_id,
           SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS feb_ctr,
           AVG(gsc_avg_position) AS feb_gsc_avg_position,
           COUNT(*) FILTER (WHERE gsc_impressions > 0) AS feb_days_with_impressions
    FROM {FACT_PRIOR}
    GROUP BY content_hash_id
""").df()

content_attrs = con.sql(f"""
    SELECT content_hash_id,
           content_type,
           DATE_DIFF('day', content_created_date, DATE '2026-03-01') AS content_age_days_at_march
    FROM {DIM_CONTENT}
""").df()

features = (feb_agg
            .merge(content_attrs, on="content_hash_id", how="inner")
            .merge(label_df[["content_hash_id", "march_ctr", "march_clicks", "march_impressions"]],
                   on="content_hash_id", how="inner"))

print(features.shape)
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(303572, 9)


,content_hash_id,feb_ctr,feb_gsc_avg_position,feb_days_with_impressions,content_type,content_age_days_at_march,march_ctr,march_clicks,march_impressions
0,content_1eea820697c3b95a,0.000000,12.946228,28,keyword article,227,0.000000,0.0,315.0
1,content_9abd8b303f805847,0.008186,6.495085,28,keyword article,227,0.000275,4.0,14536.0
2,content_5f58c55cbfee172a,0.000000,10.490023,28,keyword article,227,0.000000,0.0,387.0
3,content_6fe390ba3af1e456,0.001024,38.436254,28,keyword article,227,0.001065,5.0,4697.0
4,content_3ad5d2160242b9ca,0.002062,9.710810,28,keyword article,227,0.000996,1.0,1004.0


In [37]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

honest_feats = ["feb_ctr", "feb_gsc_avg_position", "feb_days_with_impressions", "content_age_days_at_march"]

# Fill NaN values in relevant feature and target columns with 0
features_cleaned = features.copy()
for col in ["feb_ctr", "feb_gsc_avg_position", "feb_days_with_impressions", "content_age_days_at_march", "march_ctr"]:
    features_cleaned[col] = features_cleaned[col].fillna(0)

X_honest = pd.get_dummies(features_cleaned[honest_feats + ["content_type"]], columns=["content_type"], dummy_na=True)
y = features_cleaned["march_ctr"]

Xtr, Xte, ytr, yte = train_test_split(X_honest, y, test_size=0.2, random_state=42)
model_honest = LinearRegression().fit(Xtr, ytr)
r2_honest = r2_score(yte, model_honest.predict(Xte))
print("Honest R^2 (5 features, no leakage):", round(r2_honest, 3))

X_leaky = X_honest.copy()
leaked_ctr_copy = features_cleaned["march_clicks"] / features_cleaned["march_impressions"]
leaked_ctr_copy = leaked_ctr_copy.replace([float('inf'), -float('inf')], 0).fillna(0)
X_leaky["leaked_ctr_copy"] = leaked_ctr_copy

Xtr_l, Xte_l, ytr_l, yte_l = train_test_split(X_leaky, y, test_size=0.2, random_state=42)
model_leaky = LinearRegression().fit(Xtr_l, ytr_l)
r2_leaky = r2_score(yte_l, model_leaky.predict(Xte_l))
print("Leaky R^2 (with march_clicks sneaked in):", round(r2_leaky, 3))
print("\nThe jump is the label leaking back in as a 'feature' — not real signal.")

final_features = honest_feats
print("\nFinal feature set (leak removed):", final_features)
print("Honest R^2 to report:", round(r2_honest, 3))

Honest R^2 (5 features, no leakage): 0.004
Leaky R^2 (with march_clicks sneaked in): 1.0

The jump is the label leaking back in as a 'feature' — not real signal.

Final feature set (leak removed): ['feb_ctr', 'feb_gsc_avg_position', 'feb_days_with_impressions', 'content_age_days_at_march']
Honest R^2 to report: 0.004


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [38]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
DIM_CLIENTS = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')"

history_spread = con.sql(f"""
    SELECT
        MIN(DATE_DIFF('day', gsc_data_start, DATE '2026-06-30')) AS shortest_history_days,
        MAX(DATE_DIFF('day', gsc_data_start, DATE '2026-06-30')) AS longest_history_days,
        COUNT(*) FILTER (WHERE gsc_data_start IS NULL) AS clients_with_no_start_date
    FROM {DIM_CLIENTS}
""").df()

history_spread

,shortest_history_days,longest_history_days,clients_with_no_start_date
0,28,519,37


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.